# 04. Privacy in Action: Anonymisation & De-Identification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week12/04.Privacy-in-Action-Anonymisation/notebooks/01_04.Privacy-in-Action-Anonymisation.ipynb)

## Learning Objectives
- Differentiate between **Direct Identifiers**, **Quasi-Identifiers**, and **Sensitive Attributes** under Australian Privacy Principles (APPs).
- Implement one-way salted cryptographic hashing (`hashlib.sha256`) for secure pseudonymisation.
- Coarsen and generalise quasi-identifiers (age binning and postcode prefixing).
- Audit and calculate $k$-anonymity across equivalence classes.


## 1. Classifying Privacy Attributes in Raw Data
Under the *Privacy Act 1988* and OAIC guidelines, personal information must be de-identified before release for public research.
Let's inspect an identified student counselling dataset.

In [ ]:
import pandas as pd
import numpy as np
import hashlib

raw_records = pd.DataFrame({
    'student_id': ['S1001', 'S1002', 'S1003', 'S1004', 'S1005', 'S1006', 'S1007', 'S1008'],
    'full_name': ['Alice Nguyen', 'Bob Chen', 'Charlie Smith', 'Diana Miller',
                  'Evan Davis', 'Fiona Taylor', 'George Wilson', 'Hannah Brown'],
    'age': [19, 19, 24, 25, 22, 23, 31, 35],
    'gender': ['Female', 'Female', 'Male', 'Female', 'Male', 'Female', 'Male', 'Female'],
    'postcode': ['2060', '2060', '3000', '3000', '2060', '2060', '4000', '4000'],
    'counselling_reason': [
        'Exam Stress', 'Exam Stress', 'Anxiety', 'Financial Crisis',
        'Anxiety', 'Exam Stress', 'Depression', 'Workload Stress'
    ]
})

print('--- Raw Identified Table ---')
display(raw_records)

print('Attribute Classifications:')
print('  • Direct Identifiers: student_id, full_name (Must be redacted)')
print('  • Quasi-Identifiers: age, gender, postcode (Linkage attack risk)')
print('  • Sensitive Attribute: counselling_reason (Needs privacy protection)')

## 2. Pseudonymisation via Salted Cryptographic Hashing
Simply hashing an ID with raw MD5 or SHA-256 is insecure because attackers can pre-compute rainbow tables.
We append a secret cryptographic salt before computing the SHA-256 hash.

In [ ]:
SALT = 'ACU_CONFIDENTIAL_SALT_2026'

def salted_hash(identifier: str) -> str:
    combined = f'{SALT}_{identifier}'.encode('utf-8')
    return hashlib.sha256(combined).hexdigest()[:10]

pseudo_df = raw_records.copy()
pseudo_df['pseudo_id'] = pseudo_df['student_id'].apply(salted_hash)
# Drop direct identifiers completely
pseudo_df = pseudo_df.drop(columns=['student_id', 'full_name'])

print('Pseudonymised Dataset:')
display(pseudo_df)

## 3. Generalisation & Coarsening Quasi-Identifiers
Even without names or student IDs, unique combinations of quasi-identifiers (`age`, `gender`, `postcode`) can re-identify a student.
We apply **coarsening**:
- Binning exact ages into life-stage brackets (`<=19`, `20-29`, `30+`).
- Truncating 4-digit postcodes to 2-digit regional prefixes (`20XX`, `30XX`, `40XX`).

In [ ]:
coarsened_df = pseudo_df.copy()

# Bin age
coarsened_df['age_bracket'] = pd.cut(
    coarsened_df['age'],
    bins=[0, 19, 29, 120],
    labels=['<=19', '20-29', '30+']
).astype(str)

# Truncate postcode to 2 digits
coarsened_df['postcode_prefix'] = coarsened_df['postcode'].astype(str).str[:2] + 'XX'

# Drop granular quasi-identifiers
coarsened_df = coarsened_df.drop(columns=['age', 'postcode'])

print('Coarsened Quasi-Identifiers:')
display(coarsened_df)

## 4. Evaluating $k$-Anonymity
A dataset satisfies **$k$-anonymity** if each group of quasi-identifiers contains at least $k$ distinct individuals.
Let's calculate the size of every equivalence class in our coarsened dataset.

In [ ]:
quasi_cols = ['age_bracket', 'gender', 'postcode_prefix']
equiv_classes = (
    coarsened_df.groupby(quasi_cols, as_index=False)
    .size()
    .rename(columns={'size': 'group_size'})
)

min_k = equiv_classes['group_size'].min()
print('Equivalence Classes Distribution:')
display(equiv_classes)
print(f'\nCalculated Minimum k-Anonymity: k = {min_k}')

if min_k >= 2:
    print('✅ Dataset satisfies 2-anonymity.')
else:
    print('⚠️ Privacy warning: Records with group_size = 1 are uniquely identifiable.')

## 5. Practice Exercises

### Exercise: Clinical Anonymisation
You are provided with a patient log.
1. Drop the patient name.
2. Coarsen `age` into two brackets: `18-35` and `36-60`.
3. Group by `['age_group', 'postcode']` and verify that the dataset satisfies $k \ge 2$.

In [ ]:
# Exercise Data
raw_patient_log = pd.DataFrame({
    'patient_name': ['John D', 'Sarah M', 'Ken P', 'Laura K'],
    'age': [22, 28, 45, 48],
    'postcode': ['3000', '3000', '2000', '2000'],
    'diagnosis': ['Flu', 'Asthma', 'Diabetes', 'Hypertension']
})
display(raw_patient_log)

# --- Student Solution ---
anon_clinic = raw_patient_log.drop(columns=['patient_name'])
anon_clinic['age_group'] = pd.cut(anon_clinic['age'], bins=[18, 35, 60], labels=['18-35', '36-60']).astype(str)
anon_clinic = anon_clinic.drop(columns=['age'])

clinic_equiv = anon_clinic.groupby(['age_group', 'postcode'], as_index=False).size().rename(columns={'size': 'k_count'})
print('\nClinic Anonymised Table:')
display(anon_clinic)
print('\nEquivalence Class Audit:')
display(clinic_equiv)
print(f'Clinic Achieved k-Anonymity: k = {clinic_equiv["k_count"].min()}')